# Ingest Drivers.json file
1. Read the file using Spark dataframe reader API
1. Define and enforce schema (preserve the nested structure)
1. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.Bronze-helpers


In [0]:

# Define source_file and table_name
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

### Step 1 - Read the json file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType
name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyame', StringType(),)
])

drivers_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateofBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType()),
])

In [0]:
# Read data from the drivers file
drivers_df = (
    spark.read
        .format('json')
        .schema(drivers_schema)
        .option('mode','FAILFAST')
        .load(source_file)
)



#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)


#### Step 3 - Write to bronze delta table

In [0]:
(
    drivers_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)     
)

In [0]:
display(spark.table(table_name))